# 04. Day 0 cell-type-specific differential expression

This notebook compares `Insm1_KO` versus `Control` within annotated cell types from `03_annotated.h5ad`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sns.set_style("whitegrid")

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "samples.tsv").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not find day0_insm1_scrna project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MARKERS_DIR = PROJECT_ROOT / "results" / "markers"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

INPUT_H5AD = PROCESSED_DIR / "03_annotated.h5ad"
MARKERS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input object: {INPUT_H5AD}")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
adata

In [ ]:
CELLTYPE_KEY = "manual_celltype_annotation"
CONDITION_KEY = "condition"
KO_LABEL = "Insm1_KO"
CONTROL_LABEL = "Control"
TARGET_CELLTYPES = ["Rod", "Cone", "MG"]

pd.crosstab(adata.obs[CELLTYPE_KEY], adata.obs[CONDITION_KEY])

In [ ]:
def sanitize_label(label: str) -> str:
    return label.replace(" ", "_").replace("/", "_")


def run_celltype_de(adata, cell_type):
    subset = adata[
        adata.obs[CELLTYPE_KEY].astype(str).eq(cell_type)
        & adata.obs[CONDITION_KEY].astype(str).isin([CONTROL_LABEL, KO_LABEL])
    ].copy()

    counts = subset.obs[CONDITION_KEY].value_counts()
    print(f"{cell_type}: {dict(counts)}")
    if not {CONTROL_LABEL, KO_LABEL}.issubset(set(counts.index)):
        raise ValueError(f"{cell_type} does not contain both conditions")

    sc.tl.rank_genes_groups(
        subset,
        groupby=CONDITION_KEY,
        groups=[KO_LABEL],
        reference=CONTROL_LABEL,
        method="wilcoxon",
        pts=True,
        use_raw=False,
    )
    de = sc.get.rank_genes_groups_df(subset, group=KO_LABEL)
    de = de.replace([np.inf, -np.inf], np.nan).dropna(subset=["names", "logfoldchanges", "pvals_adj"])
    de["cell_type"] = cell_type
    de["comparison"] = f"{KO_LABEL}_vs_{CONTROL_LABEL}"
    de["neg_log10_padj"] = -np.log10(de["pvals_adj"].clip(lower=1e-300))
    return subset, de


def classify_de(de, logfc_threshold=1.0, padj_threshold=0.05):
    de = de.copy()
    de["regulation"] = "Not significant"
    de.loc[(de["logfoldchanges"] >= logfc_threshold) & (de["pvals_adj"] < padj_threshold), "regulation"] = "Up in KO"
    de.loc[(de["logfoldchanges"] <= -logfc_threshold) & (de["pvals_adj"] < padj_threshold), "regulation"] = "Down in KO"
    return de


def plot_volcano(de, cell_type):
    palette = {"Up in KO": "#d73027", "Down in KO": "#4575b4", "Not significant": "#9e9e9e"}
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=de,
        x="logfoldchanges",
        y="neg_log10_padj",
        hue="regulation",
        palette=palette,
        s=12,
        linewidth=0,
        alpha=0.75,
        ax=ax,
    )
    ax.axvline(1, color="black", linestyle="--", linewidth=0.8)
    ax.axvline(-1, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.8)
    ax.set_title(f"{cell_type}: {KO_LABEL} vs {CONTROL_LABEL}")
    ax.set_xlabel("log fold change")
    ax.set_ylabel("-log10 adjusted p-value")
    plt.tight_layout()
    out_path = FIGURES_DIR / f"04_volcano_{sanitize_label(cell_type)}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved volcano plot: {out_path}")

In [ ]:
de_results = {}
subsets = {}

for cell_type in TARGET_CELLTYPES:
    if cell_type not in set(adata.obs[CELLTYPE_KEY].astype(str)):
        print(f"Skipping {cell_type}: not present in annotations")
        continue
    subset, de = run_celltype_de(adata, cell_type)
    de = classify_de(de)
    subsets[cell_type] = subset
    de_results[cell_type] = de
    out_csv = MARKERS_DIR / f"04_de_{sanitize_label(cell_type)}_{KO_LABEL}_vs_{CONTROL_LABEL}.csv"
    de.to_csv(out_csv, index=False)
    print(f"Saved DE table: {out_csv}")
    plot_volcano(de, cell_type)

In [ ]:
de_summary = []
for cell_type, de in de_results.items():
    de_summary.append({
        "cell_type": cell_type,
        "n_cells_control": int((subsets[cell_type].obs[CONDITION_KEY] == CONTROL_LABEL).sum()),
        "n_cells_ko": int((subsets[cell_type].obs[CONDITION_KEY] == KO_LABEL).sum()),
        "n_up_in_ko": int((de["regulation"] == "Up in KO").sum()),
        "n_down_in_ko": int((de["regulation"] == "Down in KO").sum()),
        "n_tested_genes": int(de.shape[0]),
    })

de_summary = pd.DataFrame(de_summary)
de_summary.to_csv(MARKERS_DIR / "04_de_summary.csv", index=False)
de_summary

In [ ]:
all_de = pd.concat(de_results.values(), ignore_index=True) if de_results else pd.DataFrame()
all_de.to_csv(MARKERS_DIR / f"04_de_all_target_celltypes_{KO_LABEL}_vs_{CONTROL_LABEL}.csv", index=False)
all_de.head()